# Spark 4.2.1 Cluster Environment Verification & Architecture Inspection

**Module**: 00_environment_and_spark_setup  
**Notebook**: 00_spark_environment.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How our 5-container Docker Spark Standalone cluster is organized, how to connect from JupyterLab, and how to verify driver-to-executor communication.
- **Why does this topic exist?**: Before executing distributed algorithms, you must verify network reachability, worker registration, event logging, and cluster resource quotas.
- **What problem does it solve?**: Prevents silent misconfigurations like wrong driver host IP, missing event logging volumes, or inactive worker executors.
- **Where is it used in real-world Data Engineering?**: Every production Spark engineer must validate cluster topology, driver binding, and memory quotas before running production workloads.

---

## 2. Theoretical Architecture of Our Docker Cluster

In our environment, Docker Compose coordinates five synchronized containers:

```
+---------------------------------------------------------------------------------------+
| Docker Host Network (127.0.0.1)                                                       |
|                                                                                       |
|   :8888  JupyterLab (Driver)  --------------------+                                   |
|   :4040  Driver Web UI                            |                                   |
|                                                   v RPC: 7077                         |
|   :8080  Spark Master <==================> Workers:                                   |
|                                            - Worker 1 (Port :8081, 2 cores, 4GB)      |
|                                            - Worker 2 (Port :8082, 2 cores, 4GB)      |
|                                                   |                                   |
|                                                   v Event Logs                        |
|   :18080 Spark History Server <=========== ./events (Shared Host Volume)             |
|                                                                                       |
|   Shared Volumes: ./data -> /opt/spark-data | ./events -> /opt/spark-events          |
+---------------------------------------------------------------------------------------+
```

### Key Service Endpoints:
- **JupyterLab (Driver)**: `http://localhost:8888/?token=spark`
- **Spark Master UI**: `http://localhost:8080` (Worker status, memory, cores)
- **Spark Worker 1 UI**: `http://localhost:8081`
- **Spark Worker 2 UI**: `http://localhost:8082`
- **Spark History Server**: `http://localhost:18080` (Completed job event logs)
- **Driver Web UI**: `http://localhost:4040` (Active while SparkSession is running)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Environment_Verification_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical Environment Inspection

Let us inspect the Python, Java, and Spark cluster runtime versions directly from code.

In [ ]:
import sys
import platform
import os

print(f"Python Executable: {sys.executable}")
print(f"Python Version:    {sys.version}")
print(f"Platform:          {platform.platform()}")

# Verify SparkContext environment
sc = spark.sparkContext
print(f"Spark App ID:      {sc.applicationId}")
print(f"Spark User:        {sc.sparkUser()}")
print(f"Default Parallelism: {sc.defaultParallelism}")

## 4. Internal Investigation: Cluster Cores and Workers

Let us check how many worker cores are registered with our driver by running a distributed RDD operation across all workers.

In [ ]:
# Run an RDD parallelize across default parallelism to check task execution
rdd = sc.parallelize(range(100), sc.defaultParallelism)
host_distribution = rdd.mapPartitions(lambda it: [os.uname().nodename]).collect()

print("Partition distribution across nodes:")
for idx, host in enumerate(host_distribution):
    print(f"  Partition {idx} ran on container: {host}")

## 5. Verifying Available Datasets in `/opt/spark-data`

Our lab includes several realistic datasets mounted from `./data` on the host.

In [ ]:
import glob

data_files = glob.glob("/opt/spark-data/**", recursive=True)
print("Available datasets mounted in /opt/spark-data:")
for f in sorted(data_files):
    if os.path.isfile(f):
        print(f"  {f} ({os.path.getsize(f):,} bytes)")

---

## Experiment: Verifying Event Logging to Persistent Storage

- **1. What are we testing?**: Confirm that running a Spark job writes an event log to /opt/spark-events so the History Server can render it.
- **2. What do we expect?**: Event log files will appear in /opt/spark-events, visible to History Server on port 18080.

In [ ]:
# Run a sample job to trigger event log write
sample_df = spark.range(0, 10000).filter("id % 2 == 0")
print(f"Sample even count: {sample_df.count()}")

event_files = os.listdir("/opt/spark-events")
print(f"Event logs found in /opt/spark-events: {len(event_files)} entries")
for ef in event_files[:5]:
    print(f"  - {ef}")

**3. What happened?**: A new event log directory was created under /opt/spark-events for this application ID.  
**4. Why did it happen?**: Because spark.eventLog.enabled=true and spark.eventLog.dir=file:///opt/spark-events are configured in the builder.  
**5. Practical Engineering Lesson**: Persistent event logging allows post-mortem diagnostics on port 18080 even after Jupyter kernel restarts.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Active Spark Configurations
**Problem Statement**: Write code to retrieve and print spark.sql.shuffle.partitions and spark.driver.host from spark.conf.  
**Input / Context**: The active spark session.

In [ ]:
# Exercise 1: Print active configs

### Exercise 2 (Intermediate): Inspect Executor Hostname Distribution
**Problem Statement**: Create an RDD with 8 partitions and collect the unique container hostnames executing the tasks.  
**Input / Context**: sc.parallelize(range(100), 8)

In [ ]:
# Exercise 2: Unique executor hostnames

### Exercise 3 (Advanced): Read and Verify Retail Orders Dataset
**Problem Statement**: Read /opt/spark-data/retail/orders.csv, count total rows, and display schema.  
**Input / Context**: /opt/spark-data/retail/orders.csv

In [ ]:
# Exercise 3: Read retail orders

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Active Spark Configurations):
```python
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.driver.host:", spark.conf.get("spark.driver.host"))
```

#### Solution 2 (Inspect Executor Hostname Distribution):
```python
hosts = sc.parallelize(range(100), 8).mapPartitions(lambda it: [os.uname().nodename]).distinct().collect()
print("Executors running on:", hosts)
```

#### Solution 3 (Read and Verify Retail Orders Dataset):
```python
df_orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
df_orders.printSchema()
print("Total orders count:", df_orders.count())
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Cluster Setup**: Our cluster has 1 Master (7077), 2 Workers (8081, 8082), 1 History Server (18080), and JupyterLab (8888).
- **Driver Host**: spark.driver.host=jupyter is required so executors in worker containers can route RPC packets back to the driver.
- **Event Logs**: Event logging persists metrics to /opt/spark-events for History Server post-mortem analysis.

### Essential Technical Terms
- `SparkContext`: Low-level connection object to the Spark cluster.
- `SparkSession`: Unified high-level entry point introduced in Spark 2.0+.
- `Default Parallelism`: Default partition count for RDD operations (equals total available cores).

### Common Engineering Mistakes
- **Driver set to localhost**: Workers cannot reach localhost because localhost resolves to the worker container itself.
- **Missing event log volume**: If ./events is not mounted, the History Server cannot view past runs.

### Interview Questions on This Topic
- **Q: Why is spark.driver.host necessary in Docker?**
  - *A*: In containerized networks, containers have individual IPs and hostnames. Setting the driver host tells workers where to send RPC heartbeats.
- **Q: What port does the active Driver UI run on?**
  - *A*: Port 4040.

### What to Study Next
Open 01_spark_fundamentals/01_introduction_to_spark.ipynb to explore core Spark concepts and primitives.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')